# Solar Prediction Colab Horizon Sweep (Hugging Face)

Train GRU and LSTM forecasters at 15-minute, 1-hour, 4-hour, and 24-hour horizons on the full dataset, with several seeds, and compare them on the test split with persistence and seasonal-naive baselines, over all hours and daylight hours only.

All horizons share one configuration. The two-stage GRU sweep (`colab_gru_tuning_actual_data.ipynb`) found that window length and model size change validation RMSE by about 1%, so this notebook uses its 24-hour winner: a 12-row window with target-time features, hidden size 32, one layer. Training uses early stopping, so `epochs` is only a cap.

Finished runs are uploaded to the Hugging Face model repo as they complete; rerunning on a fresh runtime restores them from the Hub and skips them. Test forecasts of each network's first seed are saved per horizon for the plots.

Set `HF_TOKEN` in Colab Secrets (`userdata`) or `os.environ['HF_TOKEN']` before running. Use a GPU runtime.

In [ ]:
GIT_REPO_URL = 'https://github.com/itxtx/solar_prediction.git'
PROJECT_DIR = '/content/solar_prediction'

# Hugging Face Hub repositories
HF_DATASET_REPO = 'itxtx/solar-prediction-data'
HF_DATASET_FILENAME = 'solar_weather.csv'
HF_MODEL_REPO = 'itxtx/solar-prediction-artifacts'
HF_PATH_IN_REPO = 'colab_horizon_sweep_multiseed'
UPLOAD_TO_HF = True
RESUME_FROM_HF = True

DATA_PATH = 'data/solar_weather.csv'
OUTPUT_ROOT = f'artifacts/{HF_PATH_IN_REPO}'

HORIZONS = {
    '15min': 1,
    '1h': 4,
    '4h': 16,
    '24h': 96,
}
MODELS = ['gru', 'lstm']
SEEDS = [0, 1, 2]
SEASONAL_LAG = 96  # one day of 15-minute rows

CONFIG = dict(
    window_size=12,
    target_time_features=True,
    hidden_dim=32,
    num_layers=1,
    learning_rate=1e-3,
    dropout=0.1,
    epochs=100,
    patience=10,
    batch_size=256,
)

# Example-forecast plot: horizons to show, and week start dates ('YYYY-MM-DD').
# None picks the clearest and cloudiest complete week of the test period.
EXAMPLE_HORIZONS = ['4h', '24h']
CLEAR_WEEK_START = None
CLOUDY_WEEK_START = None

In [ ]:
import os
import subprocess
from pathlib import Path

if not (Path.cwd() / 'pyproject.toml').exists():
    project = Path(PROJECT_DIR)
    if not project.exists():
        subprocess.run(['git', 'clone', '--depth', '1', GIT_REPO_URL, str(project)], check=True)
    os.chdir(project)

print('Working directory:', Path.cwd())
print('Commit:', subprocess.run(['git', 'rev-parse', 'HEAD'], capture_output=True, text=True).stdout.strip())

Use a fresh Colab runtime. This notebook installs `huggingface_hub` and the local package without touching Colab's scientific Python stack.

In [ ]:
%pip install -q huggingface_hub
%pip install -q --no-deps -e .

In [ ]:
from solar_prediction.hf_hub import (
    download_artifacts_folder,
    download_dataset_file,
    upload_artifacts_folder,
)

data_path = Path(DATA_PATH)
if not data_path.exists():
    print(f'Downloading {HF_DATASET_FILENAME} from Hugging Face dataset {HF_DATASET_REPO}...')
    data_path = download_dataset_file(
        HF_DATASET_REPO,
        HF_DATASET_FILENAME,
        local_dir=data_path.parent,
    )
assert data_path.exists(), f'Full dataset not found: {data_path}'
print('Full dataset:', data_path)

output_root = Path(OUTPUT_ROOT)
if RESUME_FROM_HF:
    restored = download_artifacts_folder(
        HF_MODEL_REPO, HF_PATH_IN_REPO, local_dir=output_root.parent
    )
    print('Restored finished runs from the Hub:', restored)
output_root.mkdir(parents=True, exist_ok=True)

## Train every horizon x network x seed

In [ ]:
import time

import pandas as pd
import torch

from solar_prediction.sweep import RunConfig, collect_runs, runs_test_metrics, summarize_test, train_run

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', device)
config = RunConfig(**CONFIG)


def upload(message):
    if UPLOAD_TO_HF:
        print('Uploaded:', upload_artifacts_folder(
            output_root, HF_MODEL_REPO, path_in_repo=HF_PATH_IN_REPO, commit_message=message
        ))


total = len(HORIZONS) * len(MODELS) * len(SEEDS)
done = 0
for horizon_label, horizon_steps in HORIZONS.items():
    for model_name in MODELS:
        trained_any = False
        for seed in SEEDS:
            done += 1
            started = time.time()
            _, trained = train_run(
                config,
                horizon_label=horizon_label,
                horizon_steps=horizon_steps,
                seed=seed,
                data_path=data_path,
                root=output_root,
                device=device,
                model=model_name,
            )
            trained_any |= trained
            status = f'trained in {time.time() - started:.0f}s' if trained else 'skipped (already done)'
            print(f'[{done}/{total}] {horizon_label} {model_name} seed{seed}: {status}')
        if trained_any:
            upload(f'Horizon sweep: {horizon_label} {model_name}')

runs = pd.concat([collect_runs(output_root, model=m) for m in MODELS], ignore_index=True)
runs = runs[(runs.run_name == config.name) & runs.horizon.isin(list(HORIZONS))]
runs.to_csv(output_root / 'runs_validation.csv', index=False)
runs.groupby(['horizon_steps', 'horizon', 'model']).val_rmse.agg(['count', 'mean', 'std'])

## Test metrics against the baselines

In [ ]:
test_runs, forecasts = runs_test_metrics(runs, data_path, seasonal_lag=SEASONAL_LAG, device=device)
test_runs.to_csv(output_root / 'test_metrics_by_seed.csv', index=False)
test_summary = summarize_test(test_runs)
test_summary.to_csv(output_root / 'test_metrics_summary.csv', index=False)

forecast_dir = output_root / 'test_forecasts'
forecast_dir.mkdir(exist_ok=True)
for horizon_label, frame in forecasts.items():
    frame.to_csv(forecast_dir / f'{horizon_label}_test_forecasts.csv', index=False)

horizon_order = list(HORIZONS)
table = test_summary.assign(
    rmse=lambda t: t.test_rmse_mean.round(2).astype(str) + ' ± ' + t.test_rmse_std.fillna(0).round(2).astype(str),
    r2=lambda t: t.test_r2_mean.round(3),
).pivot(index=['scope', 'model'], columns='horizon', values=['rmse', 'r2'])
table = table.reindex(columns=horizon_order, level=1)
table.to_csv(output_root / 'test_metrics_table.csv')
table

## Plots

In [ ]:
import matplotlib.pyplot as plt

from solar_prediction.report_plots import (
    pick_example_weeks,
    plot_example_forecasts,
    plot_rmse_by_horizon,
    plot_rmse_by_hour,
    plot_skill_by_horizon,
    skill_table,
)

networks = [m for m in ('lstm', 'gru') if m in MODELS]
figures = {
    'test_rmse_by_horizon.png': plot_rmse_by_horizon(test_summary, horizon_order, networks),
    'skill_by_horizon.png': plot_skill_by_horizon(test_summary, horizon_order, networks),
    'daylight_rmse_by_hour.png': plot_rmse_by_hour(forecasts, EXAMPLE_HORIZONS, networks),
}

auto_clear, auto_cloudy = pick_example_weeks(forecasts[EXAMPLE_HORIZONS[0]])
weeks = {
    'Clear week': pd.Timestamp(CLEAR_WEEK_START) if CLEAR_WEEK_START else auto_clear,
    'Cloudy week': pd.Timestamp(CLOUDY_WEEK_START) if CLOUDY_WEEK_START else auto_cloudy,
}
print('Example weeks:', {k: f'{v:%Y-%m-%d}' for k, v in weeks.items()})
figures['example_forecasts.png'] = plot_example_forecasts(forecasts, EXAMPLE_HORIZONS, weeks, networks)

skill_table(test_summary, networks).to_csv(output_root / 'skill_by_horizon.csv', index=False)
for name, fig in figures.items():
    fig.savefig(output_root / name, dpi=150, bbox_inches='tight')
plt.show()

upload('Horizon sweep test metrics, forecasts, and plots')